In [ ]:
"""
itr_ranking.py
==============
Baixa os dados ITR da CVM para um ano inteiro, calcula EV/EBIT e ROIC
de TODAS as empresas disponíveis e retorna um ranking por trimestre.

Uso:
    from itr_ranking import processar_ano

    df_2023 = processar_ano(2023)
    df_2023.to_csv("ranking_2023.csv", index=False, encoding="utf-8-sig")

Nota sobre EV:
    EV = Market Cap + Dívida Líquida.
    Market Cap exige preço de mercado, que não está nos arquivos CVM.
    Por isso retornamos também o ranking por ROIC puro (sem preço)
    e deixamos o EV/EBIT como coluna opcional a ser preenchida depois.

Instalação:
    pip install pandas requests
"""

import requests
import numpy as np
import pandas as pd
from io import BytesIO
from zipfile import ZipFile

BASE_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"

# Contas CVM
CONTAS_DRE = {"ebit": "3.05", "ll": "3.11", "ir_csll": "3.08"}
CONTAS_BPA = {"caixa": "1.01.01", "aplic_cp": "1.01.02"}
CONTAS_BPP = {"divida_cp": "2.01.04", "divida_lp": "2.02.01", "pl": "2.03"}

TODAS_CONTAS = {**CONTAS_DRE, **CONTAS_BPA, **CONTAS_BPP}

# ---------------------------------------------------------------------------
# 1. Download do ZIP anual e extração dos CSVs internos
# ---------------------------------------------------------------------------

def baixar_zip_ano(ano: int) -> dict[str, pd.DataFrame] | None:
    """
    Baixa itr_cia_aberta_{ano}.zip e extrai os três CSVs relevantes:
        itr_cia_aberta_DRE_con_{ano}.csv
        itr_cia_aberta_BPA_con_{ano}.csv
        itr_cia_aberta_BPP_con_{ano}.csv

    Retorna dicionário {'dre': df, 'bpa': df, 'bpp': df} ou None se falhar.
    """
    url = f"{BASE_URL}/itr_cia_aberta_{ano}.zip"
    print(f"Baixando {url} ...")

    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  ✗ Erro no download: {e}")
        return None

    alvos = {
        "dre": f"itr_cia_aberta_DRE_con_{ano}.csv",
        "bpa": f"itr_cia_aberta_BPA_con_{ano}.csv",
        "bpp": f"itr_cia_aberta_BPP_con_{ano}.csv",
    }

    resultado = {}
    with ZipFile(BytesIO(r.content)) as z:
        arquivos_zip = z.namelist()
        print(f"  Arquivos no ZIP: {arquivos_zip}")

        for chave, nome_csv in alvos.items():
            if nome_csv not in arquivos_zip:
                print(f"  ✗ {nome_csv} não encontrado no ZIP")
                continue
            df = pd.read_csv(
                z.open(nome_csv), sep=";", encoding="latin1", dtype=str
            )
            resultado[chave] = df
            print(f"  ✓ {nome_csv}: {len(df):,} linhas")

    return resultado if resultado else None


# ---------------------------------------------------------------------------
# 2. Limpeza
# ---------------------------------------------------------------------------

def _limpar(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Mantém apenas DF Consolidado (quando disponível) ou Individual
    - Remove período anterior (PENÚLTIMO)
    - Desduplicata versões do mesmo ITR (fica a mais recente)
    - Converte tipos e normaliza escala para R$
    """
    df = df.copy()

    # Mantém só ÚLTIMO exercício
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]

    # Prefere consolidado; onde não existe, aceita individual
    tem_consolidado = df[
        df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    ]["CNPJ_CIA"].unique()

    mask_cons = df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    mask_ind  = ~df["CNPJ_CIA"].isin(tem_consolidado)
    df = df[mask_cons | mask_ind]

    # Desduplicata versões — fica a mais recente
    df["VERSAO"] = pd.to_numeric(df["VERSAO"], errors="coerce")
    df = (
        df.sort_values("VERSAO", ascending=False)
          .drop_duplicates(
              subset=["CNPJ_CIA", "DT_FIM_EXERC", "CD_CONTA"],
              keep="first",
          )
    )

    # Tipos
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    df["VL_CONTA"]     = pd.to_numeric(df["VL_CONTA"], errors="coerce")

    # Normaliza escala → R$
    mask_mil = df["ESCALA_MOEDA"].str.upper().str.contains("MIL", na=False)
    df.loc[mask_mil, "VL_CONTA"] *= 1000

    return df


# ---------------------------------------------------------------------------
# 3. Pivotamento: cada CD_CONTA vira uma coluna
# ---------------------------------------------------------------------------

def _pivotar(df: pd.DataFrame, contas: dict) -> pd.DataFrame:
    """
    Filtra as contas desejadas e pivota para colunas nomeadas.
    Retorna [CNPJ_CIA, DENOM_CIA, DT_FIM_EXERC, col1, col2, ...]
    """
    mapa_inv = {v: k for k, v in contas.items()}  # CD_CONTA → nome legível

    sub = df[df["CD_CONTA"].isin(contas.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()

    piv.columns.name = None
    return piv


# ---------------------------------------------------------------------------
# 4. Cálculo dos indicadores
# ---------------------------------------------------------------------------

def _calcular(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula ROIC e as peças necessárias para EV/EBIT.
    EV/EBIT fica como NaN até que preços sejam injetados externamente.
    """
    df = df.copy()

    # --- Componentes de balanço ---
    dispon     = df.get("caixa",    pd.Series(0, index=df.index)).fillna(0) \
                + df.get("aplic_cp", pd.Series(0, index=df.index)).fillna(0)
    div_bruta  = df.get("divida_cp", pd.Series(0, index=df.index)).fillna(0) \
                + df.get("divida_lp", pd.Series(0, index=df.index)).fillna(0)
    div_liq    = div_bruta - dispon
    pl         = df.get("pl", pd.Series(np.nan, index=df.index))
    ebit       = df.get("ebit", pd.Series(np.nan, index=df.index))
    ll         = df.get("ll",   pd.Series(np.nan, index=df.index))
    ir         = df.get("ir_csll", pd.Series(0, index=df.index)).fillna(0).abs()

    df["disponibilidades"] = dispon
    df["divida_bruta"]     = div_bruta
    df["divida_liquida"]   = div_liq

    # --- Alíquota efetiva ---
    base_ir = ll.abs() + ir
    aliq = np.where(base_ir > 0, ir / base_ir, 0.34)
    df["aliquota_efetiva"] = np.clip(aliq, 0, 0.50)

    # --- NOPAT e Capital Investido ---
    df["nopat"]             = ebit * (1 - df["aliquota_efetiva"])
    cap_inv                 = pl.fillna(0) + div_liq
    df["capital_investido"] = cap_inv

    # --- ROIC ---
    df["roic"] = np.where(cap_inv.abs() > 1e-6, df["nopat"] / cap_inv, np.nan)

    # --- EV/EBIT: market_cap virá de fora; por ora NaN ---
    df["market_cap"] = np.nan   # preencher externamente se desejar
    df["ev"]         = np.nan   # market_cap + divida_liquida
    df["ev_ebit"]    = np.nan   # ev / ebit

    return df


# ---------------------------------------------------------------------------
# 5. Ranking por trimestre
# ---------------------------------------------------------------------------

def _rankear(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada trimestre (DT_FIM_EXERC), cria rankings de ROIC.
    Menor rank = melhor ROIC.
    Empresas com ROIC negativo ou nulo ficam no fim do ranking.
    """
    df = df.copy()

    df["rank_roic"] = (
        df.groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
          .astype("Int64")
    )

    # Quando EV/EBIT estiver disponível, descomentar:
    # df["rank_ev_ebit"] = (
    #     df.groupby("DT_FIM_EXERC")["ev_ebit"]
    #       .rank(ascending=True, method="min", na_option="bottom")
    #       .astype("Int64")
    # )
    # df["rank_magic"] = df["rank_roic"] + df["rank_ev_ebit"]

    return df.sort_values(["DT_FIM_EXERC", "rank_roic"])


# ---------------------------------------------------------------------------
# 6. Pipeline principal — um ano de cada vez
# ---------------------------------------------------------------------------

def processar_ano(ano: int) -> pd.DataFrame:
    """
    Pipeline completo para um ano.

    Retorna DataFrame com colunas:
        CNPJ_CIA, DENOM_CIA, DT_FIM_EXERC,
        ebit, nopat, disponibilidades, divida_bruta, divida_liquida,
        pl, capital_investido, roic, rank_roic,
        market_cap(*), ev(*), ev_ebit(*)   ← (*) NaN até injetar preços

    Exemplo de uso para vários anos:
        frames = [processar_ano(a) for a in range(2019, 2025)]
        historico = pd.concat(frames, ignore_index=True)
    """
    # Download
    dados = baixar_zip_ano(ano)
    if not dados:
        return pd.DataFrame()

    # Limpeza
    print("Limpando dados...")
    dre = _limpar(dados["dre"])
    bpa = _limpar(dados["bpa"])
    bpp = _limpar(dados["bpp"])

    # Pivotamento
    print("Pivotando contas...")
    df_dre = _pivotar(dre, CONTAS_DRE)
    df_bpa = _pivotar(bpa, CONTAS_BPA)
    df_bpp = _pivotar(bpp, CONTAS_BPP)

    # Merge dos três demonstrativos
    chave = ["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"]
    base = (
        df_dre
        .merge(df_bpa, on=chave, how="outer")
        .merge(df_bpp, on=chave, how="outer")
    )
    print(f"  {base['CNPJ_CIA'].nunique():,} empresas | "
          f"{base['DT_FIM_EXERC'].nunique()} trimestres")

    # Cálculo e ranking
    print("Calculando indicadores...")
    resultado = _calcular(base)
    resultado = _rankear(resultado)

    # Colunas finais
    cols = [
        "CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC",
        "ebit", "nopat",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "capital_investido",
        "roic", "rank_roic",
        "market_cap", "ev", "ev_ebit",
    ]
    cols_presentes = [c for c in cols if c in resultado.columns]
    resultado = resultado[cols_presentes]

    print(f"✅ Ano {ano} concluído: {len(resultado):,} linhas\n")
    return resultado

In [31]:
"""
itr_ranking.py
==============
Baixa ITR + DFP da CVM, calcula EV/EBIT e ROIC anualizados
de TODAS as empresas não-financeiras e retorna ranking por trimestre.

Correções aplicadas:
    1. DFP incluído → cobre Q4 (dezembro) para todas as empresas
    2. EBIT anualizado via DT_INI_EXERC → elimina distorção entre trimestres
    3. Setor financeiro filtrado → bancos/seguradoras excluídos
"""

import requests
import numpy as np
import pandas as pd
from io import BytesIO
from zipfile import ZipFile

# ---------------------------------------------------------------------------
# Contas CVM
# ---------------------------------------------------------------------------
CONTAS_DRE = {"ebit": "3.05", "ll": "3.11", "ir_csll": "3.08"}
CONTAS_BPA = {"caixa": "1.01.01", "aplic_cp": "1.01.02"}
CONTAS_BPP = {"divida_cp": "2.01.04", "divida_lp": "2.02.01", "pl": "2.03"}

ITR_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS"
CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"

# Setores financeiros a excluir (campo SETOR_ATIV do cadastro CVM)
SETORES_FINANCEIROS = {
    "Intermediários Financeiros",
    "Previdência e Seguros",
    "Auxiliares Financeiros",
    "Outros Intermediários Financeiros e Serviços Relacionados",
}

# ---------------------------------------------------------------------------
# 1. Download — ITR + DFP
# ---------------------------------------------------------------------------

def _ler_csv_do_zip(content: bytes, nome_csv: str) -> pd.DataFrame | None:
    try:
        with ZipFile(BytesIO(content)) as z:
            if nome_csv not in z.namelist():
                return None
            return pd.read_csv(
                z.open(nome_csv), sep=";", encoding="latin1", dtype=str
            )
    except Exception as e:
        print(f"    ✗ Erro ao ler {nome_csv}: {e}")
        return None


def baixar_dados(ano: int) -> dict[str, pd.DataFrame]:
    """
    Baixa ITR (Q1–Q3) e DFP (Q4) do ano e retorna {'dre', 'bpa', 'bpp'}.
    ITR cobre março/junho/setembro; DFP cobre dezembro (ano fiscal padrão).
    """
    frames: dict[str, list] = {"dre": [], "bpa": [], "bpp": []}
    tipos  = {"dre": "DRE_con", "bpa": "BPA_con", "bpp": "BPP_con"}

    for fonte, base_url in [("ITR", ITR_URL), ("DFP", DFP_URL)]:
        prefixo = "itr" if fonte == "ITR" else "dfp"
        zip_url = f"{base_url}/{prefixo}_cia_aberta_{ano}.zip"
        print(f"  Baixando {fonte} {ano}...", end=" ", flush=True)
        try:
            r = requests.get(zip_url, timeout=120)
            r.raise_for_status()
            print("✓")
        except Exception as e:
            print(f"✗ ({e})")
            continue

        for chave, sufixo in tipos.items():
            nome_csv = f"{prefixo}_cia_aberta_{sufixo}_{ano}.csv"
            df = _ler_csv_do_zip(r.content, nome_csv)
            if df is not None:
                df["_fonte"] = fonte   # marca origem para deduplicação posterior
                frames[chave].append(df)

    return {
        chave: pd.concat(dfs, ignore_index=True)
        for chave, dfs in frames.items()
        if dfs
    }


# ---------------------------------------------------------------------------
# 2. CNPJs do setor financeiro (para filtrar)
# ---------------------------------------------------------------------------

def _cnpjs_financeiros() -> set[str]:
    """Retorna conjunto de CNPJs de empresas em setores financeiros."""
    try:
        cad = pd.read_csv(CAD_URL, sep=";", encoding="latin1", dtype=str)
        cad.columns = cad.columns.str.strip()
        cad["CNPJ_CIA"]   = cad["CNPJ_CIA"].str.strip()
        cad["SETOR_ATIV"] = cad["SETOR_ATIV"].str.strip()
        fin = cad[cad["SETOR_ATIV"].isin(SETORES_FINANCEIROS)]
        print(f"  Setor financeiro: {len(fin):,} empresas serão excluídas")
        return set(fin["CNPJ_CIA"].unique())
    except Exception as e:
        print(f"  ⚠ Não foi possível filtrar setor financeiro: {e}")
        return set()


# ---------------------------------------------------------------------------
# 3. Limpeza
# ---------------------------------------------------------------------------

def _limpar(df: pd.DataFrame, cnpjs_financeiros: set[str]) -> pd.DataFrame:
    df = df.copy()

    df = df[~df["CNPJ_CIA"].isin(cnpjs_financeiros)]
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]

    df["DT_FIM_EXERC_dt"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    mask_itr_dez = (df["_fonte"] == "ITR") & (df["DT_FIM_EXERC_dt"].dt.month == 12)
    df = df[~mask_itr_dez].drop(columns=["DT_FIM_EXERC_dt"])

    tem_cons = df[df["GRUPO_DFP"].str.contains("Consolidado", na=False)]["CNPJ_CIA"].unique()
    mask_cons = df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    mask_ind  = ~df["CNPJ_CIA"].isin(tem_cons)
    df = df[mask_cons | mask_ind]

    df["VERSAO"] = pd.to_numeric(df["VERSAO"], errors="coerce")
    df = (
        df.sort_values("VERSAO", ascending=False)
          .drop_duplicates(subset=["CNPJ_CIA", "DT_FIM_EXERC", "CD_CONTA"], keep="first")
    )

    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    df["VL_CONTA"]     = pd.to_numeric(df["VL_CONTA"], errors="coerce")

    # ← DT_INI_EXERC só existe na DRE, não no balanço — trata ausência
    if "DT_INI_EXERC" in df.columns:
        df["DT_INI_EXERC"] = pd.to_datetime(df["DT_INI_EXERC"], errors="coerce")

    mask_mil = df["ESCALA_MOEDA"].str.upper().str.contains("MIL", na=False)
    df.loc[mask_mil, "VL_CONTA"] *= 1000

    return df.drop(columns=["_fonte"], errors="ignore")


# ---------------------------------------------------------------------------
# 4. Pivotamento
# ---------------------------------------------------------------------------

def _pivotar_dre(df: pd.DataFrame) -> pd.DataFrame:
    """
    DRE: inclui DT_INI_EXERC no índice para calcular período real depois.
    """
    mapa_inv = {v: k for k, v in CONTAS_DRE.items()}
    sub = df[df["CD_CONTA"].isin(CONTAS_DRE.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC", "DT_INI_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


def _pivotar_balanco(df: pd.DataFrame, contas: dict) -> pd.DataFrame:
    """Balanço: não precisa de DT_INI_EXERC (valores pontuais)."""
    mapa_inv = {v: k for k, v in contas.items()}
    sub = df[df["CD_CONTA"].isin(contas.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


# ---------------------------------------------------------------------------
# 5. Cálculo com EBIT anualizado
# ---------------------------------------------------------------------------

def _calcular(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Anualização: usa DT_INI_EXERC quando disponível ---
    # Após o merge outer, linhas sem correspondência na DRE terão DT_INI_EXERC NaN
    # Nesses casos assume 12 meses (fator 1.0 = sem alteração)
    if "DT_INI_EXERC" in df.columns:
        meses = (
            (df["DT_FIM_EXERC"] - df["DT_INI_EXERC"])
            / pd.Timedelta(days=30.4375)
        ).clip(lower=1).round()
        meses = meses.fillna(12)   # ← fallback para linhas sem DT_INI_EXERC
    else:
        meses = pd.Series(12, index=df.index)

    fator = (12 / meses).clip(upper=4)

    ebit_raw = df.get("ebit", pd.Series(np.nan, index=df.index))
    df["ebit_anualizado"] = ebit_raw * fator
    df["meses_periodo"]   = meses

    # resto da função continua igual, usando ebit = df["ebit_anualizado"]
    ebit = df["ebit_anualizado"]
    ll   = df.get("ll",      pd.Series(np.nan, index=df.index))
    ir   = df.get("ir_csll", pd.Series(0,      index=df.index)).fillna(0).abs()

    dispon    = df.get("caixa",     pd.Series(0, index=df.index)).fillna(0) \
              + df.get("aplic_cp",  pd.Series(0, index=df.index)).fillna(0)
    div_bruta = df.get("divida_cp", pd.Series(0, index=df.index)).fillna(0) \
              + df.get("divida_lp", pd.Series(0, index=df.index)).fillna(0)
    div_liq   = div_bruta - dispon
    pl        = df.get("pl", pd.Series(np.nan, index=df.index))

    df["disponibilidades"] = dispon
    df["divida_bruta"]     = div_bruta
    df["divida_liquida"]   = div_liq

    ll_anualizado = ll * fator
    base_ir = ll_anualizado.abs() + ir * fator
    df["aliquota_efetiva"] = np.clip(
        np.where(base_ir > 0, (ir * fator) / base_ir, 0.34), 0, 0.50
    )

    df["nopat"]             = ebit * (1 - df["aliquota_efetiva"])
    cap_inv                 = pl.fillna(0) + div_liq
    df["capital_investido"] = cap_inv
    df["roic"]              = np.where(cap_inv.abs() > 1e-6, df["nopat"] / cap_inv, np.nan)

    df["market_cap"] = np.nan
    df["ev"]         = np.nan
    df["ev_ebit"]    = np.nan

    return df

# ---------------------------------------------------------------------------
# 6. Ranking por trimestre
# ---------------------------------------------------------------------------

def _rankear(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ROIC: maior é melhor → rank 1 = maior ROIC
    df["rank_roic"] = (
        df[df["roic"] > 0]        # exclui ROIC negativos do ranking
          .groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")

    return df.sort_values(["DT_FIM_EXERC", "rank_roic"])


# ---------------------------------------------------------------------------
# 7. Pipeline principal
# ---------------------------------------------------------------------------

def processar_ano(ano: int) -> pd.DataFrame:
    """
    Pipeline completo para um ano.
    Baixa ITR (Q1–Q3) + DFP (Q4), filtra financeiras,
    anualiza EBIT e retorna ranking trimestral de todas as empresas.

    Para vários anos:
        frames = [processar_ano(a) for a in range(2019, 2025)]
        historico = pd.concat(frames, ignore_index=True)
    """
    print(f"\n{'='*55}")
    print(f"Processando {ano}")
    print(f"{'='*55}")

    # Download
    dados = baixar_dados(ano)
    if not dados:
        return pd.DataFrame()

    # CNPJs financeiros
    print("Filtrando setor financeiro...")
    cnpjs_fin = _cnpjs_financeiros()

    # Limpeza
    print("Limpando dados...")
    dre = _limpar(dados["dre"], cnpjs_fin)
    bpa = _limpar(dados["bpa"], cnpjs_fin)
    bpp = _limpar(dados["bpp"], cnpjs_fin)

    # Pivotamento
    print("Pivotando contas...")
    df_dre = _pivotar_dre(dre)
    df_bpa = _pivotar_balanco(bpa, CONTAS_BPA)
    df_bpp = _pivotar_balanco(bpp, CONTAS_BPP)

    # Merge — DRE tem DT_INI_EXERC extra
    chave_bal = ["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"]
    base = (
        df_dre
        .merge(df_bpa, on=chave_bal, how="outer")
        .merge(df_bpp, on=chave_bal, how="outer")
    )

    trimestres = sorted(base["DT_FIM_EXERC"].dropna().unique())
    print(f"  {base['CNPJ_CIA'].nunique():,} empresas | "
          f"{len(trimestres)} datas: {[str(t)[:10] for t in trimestres]}")

    # Cálculo e ranking
    print("Calculando indicadores (EBIT anualizado)...")
    resultado = _calcular(base)
    resultado = _rankear(resultado)

    # Colunas finais
    cols = [
        "CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC", "DT_INI_EXERC", "meses_periodo",
        "ebit", "ebit_anualizado", "nopat",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "capital_investido",
        "roic", "rank_roic",
        "market_cap", "ev", "ev_ebit",
    ]
    cols_presentes = [c for c in cols if c in resultado.columns]
    resultado = resultado[cols_presentes]

    print(f"✅ {ano} concluído: {len(resultado):,} linhas")
    return resultado

In [40]:
import pandas as pd
cad = pd.read_csv(
    "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv",
    sep=";", encoding="latin1", dtype=str
)
cad.columns = cad.columns.str.strip()
print(cad["SETOR_ATIV"].value_counts().head(20))

SETOR_ATIV
Máquinas, Equipamentos, Veículos e Peças      181
Construção Civil, Mat. Constr. e Decoração    172
Emp. Adm. Participações                       156
Serviços Transporte e Logística               151
Securitização de Recebíveis                   141
Metalurgia e Siderurgia                       139
Têxtil e Vestuário                            122
Energia Elétrica                              117
Bancos                                        115
Comércio (Atacado e Varejo)                   113
Alimentos                                     111
Emp. Adm. Part. - Sem Setor Principal         109
Petroquímicos e Borracha                       99
Telecomunicações                               80
Arrendamento Mercantil                         73
Emp. Adm. Part. - Energia Elétrica             53
Extração Mineral                               53
Agricultura (Açúcar, Álcool e Cana)            50
Saneamento, Serv. Água e Gás                   41
Comunicação e Informática              

In [32]:
# Um ano
#df_2023 = processar_ano(2023)

# Vários anos empilhados
frames = [processar_ano(a) for a in range(2019, 2026)]
historico = pd.concat(frames, ignore_index=True)
historico.to_csv("ranking_historico.csv", index=False, encoding="utf-8-sig")


Processando 2019
  Baixando ITR 2019... ✓
  Baixando DFP 2019... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  397 empresas | 9 datas: ['2019-01-01', '2019-02-28', '2019-03-31', '2019-05-31', '2019-06-30', '2019-08-31', '2019-09-30', '2019-11-30', '2019-12-31']
Calculando indicadores (EBIT anualizado)...
✅ 2019 concluído: 1,487 linhas

Processando 2020
  Baixando ITR 2020... ✓
  Baixando DFP 2020... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  447 empresas | 8 datas: ['2020-02-29', '2020-03-31', '2020-05-31', '2020-06-30', '2020-08-31', '2020-09-30', '2020-11-30', '2020-12-31']
Calculando indicadores (EBIT anualizado)...
✅ 2020 concluído: 1,824 linhas

Processando 2021
  Baixando ITR 2021... ✓
  Baixando DFP 2021... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  486 emp

Tendo ranking_historico.csv baixado, rodar a partir do código abaixo

In [33]:
import warnings
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz
import time
 
warnings.filterwarnings("ignore")

historico = pd.read_csv('ranking_historico.csv')

In [34]:
# ---------------------------------------------------------------------------
# 3. Preço histórico e shares via yfinance (para Market Cap e EV)
# ---------------------------------------------------------------------------
 
def _buscar_preco_shares(ticker: str, datas: list) -> pd.DataFrame:
    """
    Busca o preço de fechamento ajustado e quantidade de ações via yfinance
    para as datas dos demonstrativos.
 
    Retorna DataFrame com [DT_FIM_EXERC, preco_fechamento, shares_outstanding].
    """
    try:
        t = yf.Ticker(f"{ticker}.SA")
        info = t.info
 
        # Shares: prefere fast_info, cai para info
        shares = (
            getattr(t.fast_info, "shares", None)
            or info.get("sharesOutstanding")
        )
 
        # Histórico de preços — pega intervalo que cobre todas as datas
        datas_dt = pd.to_datetime(datas)
        start = datas_dt.min() - pd.DateOffset(days=10)
        end   = datas_dt.max() + pd.DateOffset(days=10)
 
        hist = t.history(start=start, end=end, interval="1mo", auto_adjust=True)
        if hist.empty:
            return pd.DataFrame()
 
        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)
        hist["shares"] = shares
 
        rows = []
        for data in datas_dt:
            # Preço mais próximo da data de fim do exercício
            diff = (hist["Date"] - data).abs()
            idx = diff.idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco_fechamento": hist.loc[idx, "Close"],
                "shares_outstanding": shares,
            })
 
        return pd.DataFrame(rows)
 
    except Exception as e:
        print(f"    ⚠ yfinance ({ticker}): {e}")
        return pd.DataFrame()

# ---------------------------------------------------------------------------
# 4. Cálculo dos indicadores
# ---------------------------------------------------------------------------
 
def _calcular_indicadores(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Calcula EBIT, Net Debt, Market Cap, EV e ROIC a partir das colunas extraídas.
 
    Entradas esperadas (colunas):
        ebit, caixa, aplic_cp, divida_cp, divida_lp, pl, ativo_total,
        ir_csll, ll, preco_fechamento, shares_outstanding
    """
    df = df.copy()
 
    # --- Dívida Líquida ---
    df["disponibilidades"] = df.get("caixa", 0).fillna(0) + df.get("aplic_cp", 0).fillna(0)
    df["divida_bruta"]     = df.get("divida_cp", 0).fillna(0) + df.get("divida_lp", 0).fillna(0)
    df["divida_liquida"]   = df["divida_bruta"] - df["disponibilidades"]
 
    # --- Market Cap ---
    if "preco_fechamento" in df.columns and "shares_outstanding" in df.columns:
        df["market_cap"] = df["preco_fechamento"] * df["shares_outstanding"].fillna(0)
    else:
        df["market_cap"] = np.nan
 
    # --- EV ---
    df["ev"] = df["market_cap"] + df["divida_liquida"]
 
    # --- Alíquota Efetiva de IR/CSLL ---
    # IR e CSLL costumam vir negativos na DRE (despesa)
    df["ir_csll"]  = df.get("ir_csll", pd.Series(0, index=df.index)).fillna(0).abs()
    
    df["ebt"] = (
        df.get("ebit", pd.Series(np.nan, index=df.index)) +
        df.get("resultado_financeiro", pd.Series(0, index=df.index)).fillna(0)
        )
 
    # Evita divisão por zero
    with np.errstate(divide="ignore", invalid="ignore"):
        df["aliquota_efetiva"] = np.where(
            df.get("ll", pd.Series(np.nan, index=df.index)).abs() > 0,
            df["ir_csll"] / (df.get("ll", pd.Series(np.nan, index=df.index)).abs() + df["ir_csll"]),
            0.34,   # alíquota padrão BR se não conseguir calcular
        )
        df["aliquota_efetiva"] = df["aliquota_efetiva"].clip(0, 0.50)
#     df["aliquota_efetiva"] = np.where(
#     df.get("ll", pd.Series(np.nan, index=df.index)).abs() > 0,
#     df["ir_csll"] / (df.get("ll", pd.Series(np.nan, index=df.index)).abs() + df["ir_csll"]),
#     0.34,
# )

    # Linha do NOPAT
    df["nopat"] = df.get("ebit", pd.Series(np.nan, index=df.index)) * (1 - df["aliquota_efetiva"])

    # Linha do capital investido
    df["capital_investido"] = df.get("pl", pd.Series(np.nan, index=df.index)).fillna(0) + df["divida_liquida"]

    # Linha do ROIC
    df["roic"] = np.where(
        df["capital_investido"] != 0,
        df["nopat"] / df["capital_investido"],
        np.nan,
    )

    # Linha do EV/EBIT
    df["ev_ebit"] = np.where(
        df.get("ebit", pd.Series(np.nan, index=df.index)) != 0,
        df["ev"] / df.get("ebit", pd.Series(np.nan, index=df.index)),
        np.nan,
    )
 
    df.insert(0, "ticker", ticker)
    return df


In [35]:
"""
adicionar_ev.py
===============
Adiciona market_cap, ev, ev_ebit e rank_magic ao DataFrame do itr_ranking.
"""

import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz


# ---------------------------------------------------------------------------
# Mapeamento CNPJ → ticker via brapi.dev + fuzzy match
# ---------------------------------------------------------------------------

def _buscar_todos_tickers_brapi() -> pd.DataFrame:
    url = "https://brapi.dev/api/quote/list"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        stocks = r.json().get("stocks", [])
        df = pd.DataFrame(stocks)[["stock", "name"]].copy()
        df.columns = ["ticker", "nome"]
        df["ticker"] = df["ticker"].str.upper().str.strip()
        df["nome"]   = df["nome"].str.upper().str.strip()
        df = df[df["ticker"].str.match(r'^[A-Z]{4}\d{1,2}$')]
        print(f"  brapi.dev: {len(df):,} tickers carregados")
        return df.drop_duplicates("ticker").reset_index(drop=True)
    except Exception as e:
        print(f"  ✗ Erro brapi.dev: {e}")
        return pd.DataFrame(columns=["ticker", "nome"])


def _normalizar_nome(nome: str) -> str:
    remover = [
        "S.A.", "S/A", "SA", "LTDA", "LTDA.", "S.A", "/SA",
        "CIA.", "CIA", "COMPANHIA", "PARTICIPACOES", "PARTICIPAÇÕES",
        "HOLDING", "GROUP", "BRASIL", "DO BRASIL",
        "EM RECUPERACAO JUDICIAL", "EM LIQUIDACAO EXTRAJUDICIAL",
    ]
    nome = nome.upper()
    for t in remover:
        nome = nome.replace(t, "")
    return " ".join(nome.split())


def _baixar_mapa_cnpj_ticker(df_ranking: pd.DataFrame) -> pd.DataFrame:
    empresas = (
        df_ranking[["CNPJ_CIA", "DENOM_CIA"]]
        .drop_duplicates("CNPJ_CIA")
        .dropna(subset=["DENOM_CIA"])
        .copy()
    )
    print(f"  Empresas para mapear: {len(empresas):,}")

    df_brapi = _buscar_todos_tickers_brapi()
    if df_brapi.empty:
        return pd.DataFrame(columns=["CNPJ_CIA", "TCKR"])

    nomes_brapi_norm = df_brapi["nome"].apply(_normalizar_nome).tolist()
    tickers_brapi    = df_brapi["ticker"].tolist()

    mapa = {}
    for nome in empresas["DENOM_CIA"].unique():
        nome_norm = _normalizar_nome(nome)
        match = process.extractOne(
            nome_norm, nomes_brapi_norm, scorer=fuzz.token_sort_ratio
        )
        mapa[nome] = tickers_brapi[nomes_brapi_norm.index(match[0])] \
                     if match and match[1] >= 72 else None

    empresas["TCKR"] = empresas["DENOM_CIA"].map(mapa)
    encontrados = empresas["TCKR"].notna().sum()
    print(f"  Mapeadas: {encontrados:,}/{len(empresas):,} "
          f"({len(empresas)-encontrados:,} não listadas na B3 — esperado)")

    return empresas[["CNPJ_CIA", "TCKR"]].dropna(subset=["TCKR"])


# ---------------------------------------------------------------------------
# Preços históricos via yfinance
# ---------------------------------------------------------------------------

def _buscar_precos_ticker(ticker: str, datas) -> pd.DataFrame:
    datas_dt = pd.to_datetime(datas)
    try:
        t      = yf.Ticker(f"{ticker}.SA")
        shares = getattr(t.fast_info, "shares", None) or t.info.get("sharesOutstanding")
        if not shares:
            return pd.DataFrame()

        hist = t.history(
            start=datas_dt.min() - pd.DateOffset(days=15),
            end=datas_dt.max()   + pd.DateOffset(days=15),
            interval="1mo",
            auto_adjust=True,
        )
        if hist.empty:
            return pd.DataFrame()

        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)

        rows = []
        for data in datas_dt:
            idx = (hist["Date"] - data).abs().idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco":        hist.loc[idx, "Close"],
                "shares":       shares,
            })
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame()


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------

def adicionar_ev(df_ranking: pd.DataFrame, delay: float = 0.3) -> pd.DataFrame:
    """
    Adiciona market_cap, ev, ev_ebit e rank_magic ao DataFrame do itr_ranking.
    """
    df = df_ranking.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    # --- Mapeamento CNPJ → ticker ---
    print("Construindo mapeamento CNPJ → ticker...")
    mapa = _baixar_mapa_cnpj_ticker(df)
    if mapa.empty:
        print("✗ Sem mapeamento. Abortando.")
        return df

    df = df.merge(mapa, on="CNPJ_CIA", how="left")
    print(f"  {df['TCKR'].notna().sum():,} linhas com ticker | "
          f"{df['TCKR'].isna().sum():,} sem ticker")

    # --- Busca preços: um ticker de cada vez ---
    tickers_unicos = df["TCKR"].dropna().unique()
    print(f"\nBuscando preços para {len(tickers_unicos):,} tickers...")

    frames_precos = []
    for i, ticker in enumerate(tickers_unicos):
        datas = df.loc[df["TCKR"] == ticker, "DT_FIM_EXERC"].dropna().unique()
        df_p  = _buscar_precos_ticker(ticker, datas)
        if not df_p.empty:
            df_p["TCKR"] = ticker
            frames_precos.append(df_p)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(tickers_unicos)} ({len(frames_precos)} com dados)...")
        time.sleep(delay)

    print(f"  Preços obtidos: {len(frames_precos):,} tickers")

    if not frames_precos:
        print("✗ Nenhum preço obtido.")
        return df.drop(columns=["TCKR"], errors="ignore")

    # --- Merge de preços: substitui o loop linha a linha ---
    # Constrói um DataFrame único com todos os preços e faz merge por (TCKR, DT_FIM_EXERC)
    df_precos = pd.concat(frames_precos, ignore_index=True)
    df_precos["DT_FIM_EXERC"] = pd.to_datetime(df_precos["DT_FIM_EXERC"])

    df = df.merge(
        df_precos[["TCKR", "DT_FIM_EXERC", "preco", "shares"]],
        on=["TCKR", "DT_FIM_EXERC"],
        how="left",
    )

    # Mantém ticker como coluna nomeada antes de dropar TCKR
    df = df.rename(columns={"TCKR": "ticker"})

    # --- Indicadores de mercado ---
    df["market_cap"] = df["preco"] * df["shares"]
    df["ev"]         = df["market_cap"] + df["divida_liquida"]
    df["ev_ebit"]    = np.where(
        df["ebit"].notna() & (df["ebit"] != 0),
        df["ev"] / df["ebit"],
        np.nan,
    )

    # --- Rankings ---
    # EV/EBIT: só valores positivos fazem sentido (EBIT e EV positivos)
    df["rank_ev_ebit"] = (
        df[df["ev_ebit"] > 0]
          .groupby("DT_FIM_EXERC")["ev_ebit"]
          .rank(ascending=True, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")

    df["rank_magic"] = np.where(
        df["rank_roic"].notna() & df["rank_ev_ebit"].notna(),
        df["rank_roic"].astype(float) + df["rank_ev_ebit"].astype(float),
        np.nan,
    )

    # --- Ordena por rank_magic (menor = melhor) dentro de cada trimestre ---
    df = df.sort_values(
        ["DT_FIM_EXERC", "rank_magic"],
        ascending=[True, True],
        na_position="last",
    ).reset_index(drop=True)

    # --- Reordena colunas: ticker logo após DENOM_CIA ---
    cols = df.columns.tolist()
    for col in ["ticker", "rank_roic", "rank_ev_ebit", "rank_magic"]:
        if col in cols:
            cols.remove(col)
    idx = cols.index("DENOM_CIA")
    cols = (
        cols[:idx + 1]
        + ["ticker"]
        + cols[idx + 1:]
        + ["rank_roic", "rank_ev_ebit", "rank_magic"]
    )
    df = df[[c for c in cols if c in df.columns]]

    preenchidos = df["market_cap"].notna().sum()
    print(f"\n✅ market_cap preenchido: {preenchidos:,}/{len(df):,} "
          f"({100*preenchidos/len(df):.1f}%)")
    return df

In [36]:
import pandas as pd
df_cad = pd.read_csv(
    "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv",
    sep=";", encoding="latin1", dtype=str
)
df_cad.columns = df_cad.columns.str.strip()
print(df_cad.columns.tolist())
print(df_cad.head(3))

['CNPJ_CIA', 'DENOM_SOCIAL', 'DENOM_COMERC', 'DT_REG', 'DT_CONST', 'DT_CANCEL', 'MOTIVO_CANCEL', 'SIT', 'DT_INI_SIT', 'CD_CVM', 'SETOR_ATIV', 'TP_MERC', 'CATEG_REG', 'DT_INI_CATEG', 'SIT_EMISSOR', 'DT_INI_SIT_EMISSOR', 'CONTROLE_ACIONARIO', 'TP_ENDER', 'LOGRADOURO', 'COMPL', 'BAIRRO', 'MUN', 'UF', 'PAIS', 'CEP', 'DDD_TEL', 'TEL', 'DDD_FAX', 'FAX', 'EMAIL', 'TP_RESP', 'RESP', 'DT_INI_RESP', 'LOGRADOURO_RESP', 'COMPL_RESP', 'BAIRRO_RESP', 'MUN_RESP', 'UF_RESP', 'PAIS_RESP', 'CEP_RESP', 'DDD_TEL_RESP', 'TEL_RESP', 'DDD_FAX_RESP', 'FAX_RESP', 'EMAIL_RESP', 'CNPJ_AUDITOR', 'AUDITOR']
             CNPJ_CIA                                       DENOM_SOCIAL  \
0  08.773.135/0001-00          2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL   
1  11.396.633/0001-87                        3A COMPANHIA SECURITIZADORA   
2  01.547.749/0001-16  521 PARTICIPAÇOES S.A. - EM LIQUIDAÇÃO EXTRAJU...   

                         DENOM_COMERC      DT_REG    DT_CONST   DT_CANCEL  \
0                     2W ECOBANK

In [37]:
historico_completo = adicionar_ev(historico)

Construindo mapeamento CNPJ → ticker...
  Empresas para mapear: 577
  brapi.dev: 1,269 tickers carregados
  Mapeadas: 290/577 (287 não listadas na B3 — esperado)
  8,047 linhas com ticker | 5,699 sem ticker

Buscando preços para 258 tickers...
  50/258 (50 com dados)...
  100/258 (100 com dados)...
  150/258 (149 com dados)...
  200/258 (198 com dados)...
  250/258 (247 com dados)...
  Preços obtidos: 255 tickers

✅ market_cap preenchido: 7,961/13,746 (57.9%)


In [38]:
historico_completo.to_csv("ranking.csv", index=False, encoding="utf-8-sig")

historico_completo

,CNPJ_CIA,DENOM_CIA,ticker,DT_FIM_EXERC,DT_INI_EXERC,meses_periodo,ebit,ebit_anualizado,nopat,disponibilidades,...,capital_investido,roic,market_cap,ev,ev_ebit,preco,shares,rank_roic,rank_ev_ebit,rank_magic
0,07.857.093/0001-14,AURA MINERALS INC.,AURA33,2019-01-01,2019-01-01,1.0,1.281260e+08,5.125040e+08,3.702551e+08,1.576010e+08,...,8.040200e+08,0.460505,3.403187e+09,3.418721e+09,26.682492,13.259344,2.566633e+08,1.0,1,2.0
1,64.904.295/0001-03,CAMIL ALIMENTOS S.A.,CAML3,2019-02-28,2018-03-01,12.0,3.819810e+08,3.819810e+08,3.781978e+08,3.965440e+08,...,3.201430e+09,0.118134,1.751412e+09,2.783727e+09,7.287605,5.135032,3.410712e+08,1.0,1,2.0
2,42.278.473/0001-03,WIZ CO PARTICIPAÇÕES E CORRETAGEM DE SEGUROS S.A.,WIZC3,2019-03-31,2019-01-01,3.0,8.613200e+07,3.445280e+08,2.267385e+08,1.151550e+08,...,1.183430e+08,1.915943,7.231953e+08,6.080403e+08,7.059400,4.522591,1.599073e+08,4.0,8,12.0
3,45.242.914/0001-05,C&A MODAS S.A.,CEAB3,2019-03-31,2019-01-01,3.0,6.108330e+08,2.443332e+09,1.611885e+09,1.348840e+08,...,3.418199e+09,0.471560,4.579526e+09,6.133215e+09,10.040740,15.145722,3.023643e+08,14.0,11,25.0
4,88.610.191/0001-54,MUNDIAL S.A. - PRODUTOS DE CONSUMO,MNDL3,2019-03-31,2019-01-01,3.0,1.867800e+07,7.471200e+07,3.735600e+07,2.022000e+06,...,4.816400e+07,0.775600,7.909586e+07,2.614699e+08,13.998815,7.975000,9.917976e+06,7.0,23,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13741,84.683.374/0001-49,TUPY S.A.,TUPY3,2025-12-31,2025-01-01,12.0,-1.564810e+08,-1.564810e+08,-1.196885e+08,1.853156e+09,...,4.754936e+09,-0.025171,1.637204e+09,3.878764e+09,-24.787444,12.490000,1.310812e+08,NaN,<NA>,NaN
13742,84.683.671/0001-94,WETZEL S.A.,MWET4,2025-12-31,2025-01-01,12.0,-4.120000e+06,-4.120000e+06,-2.452383e+06,3.978000e+06,...,1.144540e+08,-0.021427,1.773999e+07,1.255880e+08,-30.482521,8.620000,2.058003e+06,NaN,<NA>,NaN
13743,89.463.822/0001-12,LUPATECH S.A,LUPA3,2025-12-31,2025-01-01,12.0,-6.187800e+07,-6.187800e+07,-6.173540e+07,3.840000e+05,...,2.522340e+08,-0.244754,5.555753e+07,2.333215e+08,-3.770670,1.190000,4.668700e+07,NaN,<NA>,NaN
13744,90.400.888/0001-42,BCO SANTANDER (BRASIL) S.A.,SANB11,2025-12-31,2025-01-01,12.0,1.672899e+10,1.672899e+10,1.672899e+10,2.023273e+10,...,-8.428247e+09,-1.984872,1.650713e+11,1.448386e+11,8.657937,34.969612,4.720422e+09,NaN,110,NaN


In [39]:
import pandas as pd

def extrair_top_n_empresas(caminho_arquivo, n=15):
    # 1. Carregar os dados
    df = pd.read_csv(caminho_arquivo)
    
    # 2. Converter a coluna de data para o tipo datetime
    df['DT_FIM_EXERC'] = pd.to_datetime(df['DT_FIM_EXERC'])
    
    # 3. Criar uma coluna com o trimestre correspondente (Ex: 2019Q1)
    df['Trimestre'] = df['DT_FIM_EXERC'].dt.to_period('Q')
    
    # 4. Filtrar pelo período desejado (2019Q1 a 2024Q4)
    df = df[(df['Trimestre'] >= '2019Q1') & (df['Trimestre'] <= '2024Q4')]
    
    # 5. Ordenar pelo trimestre e pelo rank_magic (ordem crescente: 1, 2, 3...)
    df_sorted = df.sort_values(by=['Trimestre', 'rank_magic'], ascending=[True, True])
    
    # 6. Selecionar as N melhores de cada trimestre
    top_n = df_sorted.groupby('Trimestre').head(n).copy()
    
    # 7. Criar uma coluna enumerando a posição (1 a N)
    top_n['Posicao'] = top_n.groupby('Trimestre').cumcount() + 1
    
    # 8. Pivotar a tabela: Trimestres nas linhas e posições nas colunas
    df_pivot = top_n.pivot(index='Trimestre', columns='Posicao', values='ticker')
    
    # Renomear colunas para ficar mais legível
    df_pivot.columns = [f'Top_{i}' for i in df_pivot.columns]
    
    return df_pivot

# Executando a função para o Top 5 empresas por trimestre
resultado_df = extrair_top_n_empresas('ranking.csv', n=15)

# Opcional: salvar em um novo arquivo CSV
resultado_df.to_csv('melhores_por_trimestre.csv')

resultado_df

,Top_1,Top_2,Top_3,Top_4,Top_5,Top_6,Top_7,Top_8,Top_9,Top_10,Top_11,Top_12,Top_13,Top_14,Top_15
Trimestre,,,,,,,,,,,,,,,
2019Q1,AURA33,CAML3,WIZC3,CEAB3,MNDL3,EPAR3,WHRL4,AZUL3,ALUP11,GEPA3,AFLT3,AGRO3,ROMI3,BRKM5,SEER3
2019Q2,CAML3,EPAR3,WIZC3,AZUL3,WHRL4,MNDL3,GEPA3,CEAB3,TASA4,ALUP11,AGRO3,TIMS3,CAMB3,PTNT4,SEER3
2019Q3,CAML3,WIZC3,AZUL3,BBSE3,MNDL3,UNIP6,EPAR3,TGMA3,GEPA3,LEVE3,AFLT3,RANI3,TASA4,WHRL4,ALUP11
2019Q4,CAML3,RSUL4,WIZC3,MNDL3,AFLT3,EPAR3,CMIN3,GEPA3,CURY3,TGMA3,LEVE3,CEAB3,UNIP6,CGRA4,ITSA4
2020Q1,CAML3,RSUL4,PATI3,WIZC3,GEPA3,DEXP3,CLSC4,RAIZ4,TASA4,AFLT3,CAMB3,WHRL4,TAEE11,BBSE3,MNDL3
2020Q2,CAML3,RSUL4,EPAR3,DEXP3,TASA4,MTSA4,WIZC3,BALM4,NUTR3,JHSF3,HOSI11,CMIN3,SEER3,UNIP6,SLCE3
2020Q3,CAML3,PEAB4,WIZC3,RSUL4,UNIP6,WHRL4,MTRE3,EPAR3,BRAP4,CMIN3,VALE3,TASA4,CYRE3,ALLD3,LPSB3
2020Q4,CAML3,PEAB4,WIZC3,GEPA3,BOBR4,CMIN3,EPAR3,WHRL4,UNIP6,LAVV3,MNDL3,PATI3,DEXP3,CAML3,MTSA4
2021Q1,CAML3,EPAR3,BRAP4,UNIP6,CMIN3,WIZC3,VALE3,CSNA3,BRKM5,PATI3,MNDL3,RSUL4,AGRO3,WHRL4,LPSB3
